# ABAW8 EMI — Face-only model reproduction (Savchenko, CVPRW 2025)

This notebook reproduces the **face-only** branch of the model proposed in:

> A. V. Savchenko. *Leveraging Lightweight Facial Models and Textual Modality in Audio-visual Emotional Mimicry Intensity Estimation.* CVPRW 2025 (ABAW 8 challenge).
> Paper: https://openaccess.thecvf.com/content/CVPR2025W/ABAW/papers/Savchenko_Leveraging_Lightweight_Facial_Models_and_Textual_Modality_in_Audio-visual_Emotional_CVPRW_2025_paper.pdf

Reference upstream training notebooks (used as the source of truth for the recipe):
https://github.com/sb-ai-lab/EmotiEffLib/tree/main/training_and_examples/ABAW/ABAW8

The paper reports a validation Pearson correlation of **≈ 0.1776** on the EMI track using a lightweight EmotiEffNet-B0 face encoder followed by a shallow head trained on aggregated per-video embeddings. Here we follow the same recipe on top of features that are already extracted in this project by `additional_experements/face_encoding_additional_encoders.ipynb`.

**Pipeline summary**
1. Load per-video face embeddings produced by frozen `enet_b0_8_best_afew` (1280-dim, 5 fps).
2. Aggregate per video by temporal **mean + std pooling** of valid (face-detected) frames → 2560-dim feature.
3. Z-score normalise features and z-score normalise the 6 EMI emotion targets.
4. Train a small MLP head with the upstream loss `1 − Pearson(pred, target) + λ · SmoothL1`.
5. Report validation Pearson correlation per emotion and the overall mean (target ≈ 0.1776).

The presentation style mirrors `single_mod_models/image_model.ipynb` so this notebook can be read side-by-side with the project's local face model.

## 1. Configuration

In [ ]:
from pathlib import Path

# ===== Paths =====
DATA_DIR  = Path("/home/danila/networks/data")
TRAIN_CSV = DATA_DIR / "train_split.csv"
VALID_CSV = DATA_DIR / "valid_split.csv"

# EmotiEffNet-B0 face embeddings (matches the paper's backbone choice).
# Produced by `additional_experements/face_encoding_additional_encoders.ipynb`
# with EMOTIEFF_MODEL = "enet_b0_8_best_afew".
FACE_DIR  = DATA_DIR / "embeddings" / "faces_emotiefflib_enet_b0_8_best_afew_fps5_v1"

RUN_DIR   = DATA_DIR / "runs" / "abaw8_emi_face_repro"
RUN_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH = RUN_DIR / "best_by_corr.pt"
HIST_PATH = RUN_DIR / "history.json"
STATS_PATH = RUN_DIR / "feat_stats.npz"

# ===== Task =====
EMOTIONS = ["Admiration", "Amusement", "Determination", "Empathic Pain", "Excitement", "Joy"]
NUM_TARGETS = 6
ID_WIDTH = 5

MIN_VALID_FRAMES = 5

# ===== Training (matches the upstream EMI recipe) =====
DEVICE = "cuda"
SEED = 42

BATCH_SIZE = 64
GRAD_CLIP = 1.0
HIDDEN_DIM = 128
DROPOUT = 0.5

LR = 3e-4
WEIGHT_DECAY = 1e-2
MAX_EPOCHS = 80
PATIENCE = 15
MIN_DELTA = 1e-4
LAMBDA_SMOOTH = 0.02

In [ ]:
import os, json, math, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print("CUDA:", torch.cuda.is_available(), "| DEVICE:", DEVICE)
print("FACE_DIR exists:", FACE_DIR.exists(), "->", FACE_DIR)

## 2. Load embeddings and aggregate per video

The upstream EMI notebooks turn each video into a single fixed-size descriptor by pooling frame-level EmotiEffNet features over the time axis. We use **mean + std** pooling over frames where MTCNN detected a face — this is the recipe described in the paper and used in `EmotiEffLib/training_and_examples/ABAW/ABAW8/emi.ipynb`.

In [ ]:
def load_face_npz(path: Path):
    d = np.load(path, allow_pickle=False)
    emb = d["embeddings"].astype(np.float32)  # (T, D)
    if "face_found" in d.files:
        valid = d["face_found"].astype(bool)
    elif "valid" in d.files:
        valid = d["valid"].astype(bool)
    else:
        valid = np.ones((emb.shape[0],), dtype=bool)
    return emb, valid

def pool_video(emb: np.ndarray, valid: np.ndarray) -> np.ndarray:
    """Mean + std pooling over valid frames -> [2*D] vector."""
    x = emb[valid]
    if x.shape[0] == 0:
        x = emb  # fallback if no face was detected anywhere
    mean = x.mean(axis=0)
    std = x.std(axis=0) if x.shape[0] > 1 else np.zeros_like(mean)
    return np.concatenate([mean, std], axis=0).astype(np.float32)

def build_features(df: pd.DataFrame):
    feats, ys, kept_ids, missing = [], [], [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Pool face emb"):
        vid = str(row["Filename"]).zfill(ID_WIDTH)
        p = FACE_DIR / f"{vid}.npz"
        if not p.exists():
            missing.append({"video_id": vid, "reason": "missing_npz"})
            continue
        try:
            emb, valid = load_face_npz(p)
        except Exception as e:
            missing.append({"video_id": vid, "reason": f"read_error:{e!r}"})
            continue
        if valid.sum() < MIN_VALID_FRAMES:
            missing.append({"video_id": vid, "reason": "too_few_valid_frames"})
            continue
        feats.append(pool_video(emb, valid))
        ys.append(np.array([row[e] for e in EMOTIONS], dtype=np.float32))
        kept_ids.append(vid)
    X = np.stack(feats, axis=0) if feats else np.zeros((0, 0), np.float32)
    Y = np.stack(ys, axis=0) if ys else np.zeros((0, NUM_TARGETS), np.float32)
    return X, Y, kept_ids, pd.DataFrame(missing)

In [ ]:
train_df = pd.read_csv(TRAIN_CSV, dtype={"Filename": str})
valid_df = pd.read_csv(VALID_CSV, dtype={"Filename": str})
train_df["Filename"] = train_df["Filename"].str.zfill(ID_WIDTH)
valid_df["Filename"] = valid_df["Filename"].str.zfill(ID_WIDTH)

X_train, Y_train, train_ids, train_miss = build_features(train_df)
X_val,   Y_val,   val_ids,   val_miss   = build_features(valid_df)

print("Train:", X_train.shape, Y_train.shape, "missing:", len(train_miss))
print("Val  :", X_val.shape,   Y_val.shape,   "missing:", len(val_miss))

pd.concat([train_miss.assign(split="train"), val_miss.assign(split="val")]) \
  .to_csv(RUN_DIR / "missing_faces.csv", index=False)

## 3. Feature and target normalisation

Both the input features and the regression targets are z-scored using the **train split** statistics, exactly as in the upstream pipeline. Predictions are denormalised before computing the reported Pearson correlation.

In [ ]:
feat_mean = X_train.mean(axis=0)
feat_std  = np.clip(X_train.std(axis=0), 1e-6, None)

y_mean = Y_train.mean(axis=0)
y_std  = np.clip(Y_train.std(axis=0), 1e-3, None)

np.savez(STATS_PATH,
         feat_mean=feat_mean.astype(np.float32),
         feat_std=feat_std.astype(np.float32),
         y_mean=y_mean.astype(np.float32),
         y_std=y_std.astype(np.float32),
         emotions=np.array(EMOTIONS))

def zscore_x(x):
    return ((x - feat_mean) / feat_std).astype(np.float32)

def zscore_y(y):
    return ((y - y_mean) / y_std).astype(np.float32)

X_train_n = zscore_x(X_train)
X_val_n   = zscore_x(X_val)
Y_train_n = zscore_y(Y_train)
Y_val_n   = zscore_y(Y_val)

print("feature dim:", X_train_n.shape[1])
print("y_mean:", y_mean)
print("y_std :", y_std)

## 4. Dataset and DataLoader

In [ ]:
class PooledFaceDataset(Dataset):
    def __init__(self, X, Y, ids):
        self.X = torch.from_numpy(X).float()
        self.Y = torch.from_numpy(Y).float()
        self.ids = ids
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return {"x": self.X[idx], "y": self.Y[idx], "video_id": self.ids[idx]}

train_ds = PooledFaceDataset(X_train_n, Y_train_n, train_ids)
val_ds   = PooledFaceDataset(X_val_n,   Y_val_n,   val_ids)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print("Train batches:", len(train_loader), "| Val batches:", len(val_loader))

## 5. Model

The Savchenko 2025 paper uses a *shallow* head on top of frozen EmotiEffNet features. We implement it as a small MLP: `Linear → BatchNorm → ReLU → Dropout → Linear`, predicting the 6 normalised EMI intensities.

In [ ]:
class EmiFaceHead(nn.Module):
    def __init__(self, in_dim: int, hidden: int, num_targets: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_targets),
        )
    def forward(self, x):
        return self.net(x)

DIN = X_train_n.shape[1]
model = EmiFaceHead(DIN, HIDDEN_DIM, NUM_TARGETS, DROPOUT).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
print(model)

## 6. Loss and Pearson metric

Loss is `1 − mean Pearson + λ · SmoothL1`, identical to the loss used by `single_mod_models/image_model.ipynb` and to the upstream EMI training script.

In [ ]:
def pearson_corr_torch(preds, targets, eps=1e-8):
    vx = preds - preds.mean(0)
    vy = targets - targets.mean(0)
    corr = (vx * vy).sum(0) / (torch.sqrt((vx ** 2).sum(0) * (vy ** 2).sum(0)) + eps)
    return corr  # per-target [num_targets]

smooth_loss_fn = nn.SmoothL1Loss(beta=0.2)

def compute_loss(preds, targets):
    corr_per_dim = pearson_corr_torch(preds, targets)
    loss_main = 1.0 - corr_per_dim.mean()
    loss = loss_main + LAMBDA_SMOOTH * smooth_loss_fn(preds, targets)
    return loss, corr_per_dim.detach()

## 7. Training loop

Because the loss itself is a Pearson correlation it is not separable across mini-batches, so we accumulate predictions over the whole epoch and step the optimiser per batch using the in-batch correlation (this is the same trick used by the local image model notebook with `ACCUM_STEPS`). For evaluation we always compute the correlation over the full split.

In [ ]:
@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    P, Y = [], []
    for batch in loader:
        x = batch["x"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        p = model(x)
        P.append(p.float().cpu()); Y.append(y.float().cpu())
    P = torch.cat(P, 0); Y = torch.cat(Y, 0)
    corr_per_dim = pearson_corr_torch(P, Y)
    loss = 1.0 - corr_per_dim.mean().item() + LAMBDA_SMOOTH * smooth_loss_fn(P, Y).item()
    return {"loss": float(loss), "corr": float(corr_per_dim.mean()), "corr_per_dim": corr_per_dim.numpy()}

def train_epoch(model, loader, optimizer):
    model.train()
    losses, corrs = [], []
    for batch in loader:
        x = batch["x"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        p = model(x)
        loss, corr_per_dim = compute_loss(p, y)
        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            continue
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        losses.append(loss.item())
        corrs.append(corr_per_dim.mean().item())
    return {"loss": float(np.mean(losses)) if losses else float("nan"),
            "corr": float(np.mean(corrs)) if corrs else float("nan")}


In [ ]:
history = {"train_loss": [], "train_corr": [], "val_loss": [], "val_corr": [], "lr": []}
best_corr = -1e9
best_epoch = -1
bad = 0

for epoch in range(1, MAX_EPOCHS + 1):
    tr = train_epoch(model, train_loader, optimizer)
    va = eval_epoch(model, val_loader)
    scheduler.step()

    lr_now = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(tr["loss"])
    history["train_corr"].append(tr["corr"])
    history["val_loss"].append(va["loss"])
    history["val_corr"].append(va["corr"])
    history["lr"].append(lr_now)

    print(f"Epoch {epoch:03d} | train loss {tr['loss']:.4f} corr {tr['corr']:.4f} "
          f"| val loss {va['loss']:.4f} corr {va['corr']:.4f} | lr {lr_now:.2e}")

    if (va["corr"] - best_corr) > MIN_DELTA:
        best_corr = va["corr"]
        best_epoch = epoch
        bad = 0
        torch.save({"epoch": epoch, "model_state": model.state_dict(), "best_corr": best_corr,
                    "in_dim": DIN}, CKPT_PATH)
        with open(HIST_PATH, "w", encoding="utf-8") as f:
            json.dump(history, f, ensure_ascii=False, indent=2)
        print(f"  saved best-by-corr: epoch={epoch}, val_corr={best_corr:.4f}")
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f"early stopping. best epoch={best_epoch}, best val corr={best_corr:.4f}")
            break

## 8. Final evaluation and reporting

We load the best checkpoint and report:
- per-emotion validation Pearson correlation (table),
- the **mean Pearson correlation across the 6 EMI dimensions** — this is the metric reported by the paper (target ≈ 0.1776),
- training curves matching the visual style of `single_mod_models/image_model.ipynb`.

In [ ]:
import matplotlib.pyplot as plt

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print("Loaded best epoch:", ckpt["epoch"], "best_corr:", ckpt["best_corr"])

train_metrics = eval_epoch(model, train_loader)
val_metrics   = eval_epoch(model, val_loader)

per_dim = pd.DataFrame({
    "emotion": EMOTIONS,
    "val_pearson": val_metrics["corr_per_dim"],
    "train_pearson": train_metrics["corr_per_dim"],
})
print(per_dim.to_string(index=False))
print()
print(f"Mean validation Pearson : {val_metrics['corr']:.4f}")
print(f"Paper target (Savchenko CVPRW 2025): 0.1776")

In [ ]:
epochs = np.arange(1, len(history["train_loss"]) + 1)

plt.figure()
plt.plot(epochs, history["train_loss"], label="train loss (1-corr+λ)")
plt.plot(epochs, history["val_loss"],   label="val loss (1-corr+λ)")
plt.axvline(ckpt["epoch"], linestyle="--", label=f"best epoch={ckpt['epoch']}")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Loss"); plt.legend(); plt.show()

plt.figure()
plt.plot(epochs, history["train_corr"], label="train corr")
plt.plot(epochs, history["val_corr"],   label="val corr")
plt.axhline(0.1776, color="red", linestyle=":", label="paper 0.1776")
plt.axvline(ckpt["epoch"], linestyle="--", label=f"best epoch={ckpt['epoch']}")
plt.xlabel("epoch"); plt.ylabel("pearson corr"); plt.title("Validation Pearson vs paper target"); plt.legend(); plt.show()

## 9. Notes on the reproduction

* **Backbone.** EmotiEffNet-B0 (`enet_b0_8_best_afew`) — exactly the lightweight face encoder used by the paper. Embeddings are extracted once and frozen; only the head is trained here.
* **Aggregation.** Per-video temporal **mean + std pooling** of valid frames. This collapses variable-length videos into a single 2560-dim descriptor and matches the upstream EMI feature pipeline.
* **Head.** Shallow MLP (Linear → BN → ReLU → Dropout → Linear). This is intentionally simpler than the TCN-based `single_mod_models/image_model.ipynb` because the paper relies on the strength of the frozen face features rather than on a heavy temporal model.
* **Loss / metric.** `1 − Pearson + λ · SmoothL1` with λ = 0.02, mean Pearson across the 6 EMI dimensions as the headline metric — the same convention as the ABAW8 EMI track.
* **Expected result.** The face-only branch in the paper reports **≈ 0.1776** mean Pearson on validation. The full multimodal model reported in the paper combines this branch with audio (HuBERT/WavLM) and text (RoBERTa) features via late fusion.